In [55]:
# Libraries
import sys
sys.path.append('..')

import joblib

from src.utils import section

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.layers import SimpleRNN, LSTM

In [2]:
# Load
items = {
    "X_train_pad": None,
    "X_val_pad": None,
    "X_test_pad": None,
    "y_train": None,
    "y_val": None,
    "y_test": None,
    "vocab": None,
}

section(f"Shape:", 25)
for name, obj in items.items():
    items[name] = joblib.load(f"../data/processed/{name}.pkl")
    print(name, len(items[name]) if name == "vocab" else items[name].shape)

X_train_pad, X_val_pad, X_test_pad, y_train, y_val, y_test, vocab = items.values()


                         
SHAPE:
X_train_pad (63871, 256)
X_val_pad (15968, 256)
X_test_pad (8006, 256)
y_train (63871,)
y_val (15968,)
y_test (8006,)
vocab 20002


In [4]:
# import pandas as pd
# from src.text_processing import clean_text, encode, pad_id


# def predict_review(text):
#     encode_text = encode(pd.Series(text), vocab)
#     pad_text = pad_id(encode_text)
#     return model.predict(pad_text, verbose=0)

In [5]:
from src.model import train_model

# Train RNN
rnn_model, rnn_history = train_model(vocab, SimpleRNN, X_train_pad, y_train, X_val_pad, y_val)

Epoch 1/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 40s 20ms/step - accuracy: 0.8395 - loss: 0.3702 - val_accuracy: 0.8858 - val_loss: 0.2809
Epoch 2/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 39s 20ms/step - accuracy: 0.9151 - loss: 0.2220 - val_accuracy: 0.9011 - val_loss: 0.2589
Epoch 3/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 39s 20ms/step - accuracy: 0.9319 - loss: 0.1811 - val_accuracy: 0.9064 - val_loss: 0.2364
Epoch 4/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 40s 20ms/step - accuracy: 0.9385 - loss: 0.1696 - val_accuracy: 0.9043 - val_loss: 0.2654
Epoch 5/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 40s 20ms/step - accuracy: 0.9439 - loss: 0.1545 - val_accuracy: 0.9016 - val_loss: 0.2705


In [6]:
# Train LSTM
lstm_model, lstm_history = train_model(vocab, LSTM, X_train_pad, y_train, X_val_pad, y_val)

Epoch 1/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 65s 32ms/step - accuracy: 0.8795 - loss: 0.2930 - val_accuracy: 0.9186 - val_loss: 0.2162
Epoch 2/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 63s 32ms/step - accuracy: 0.9317 - loss: 0.1850 - val_accuracy: 0.9080 - val_loss: 0.2400
Epoch 3/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 64s 32ms/step - accuracy: 0.9449 - loss: 0.1517 - val_accuracy: 0.9064 - val_loss: 0.2539


In [23]:
# Train TRANSFORMER
from src.model import build_transformer

transformer_model, transformer_history = build_transformer(vocab, X_train_pad, y_train, X_val_pad, y_val)

Epoch 1/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 51s 25ms/step - accuracy: 0.8746 - loss: 0.3234 - val_accuracy: 0.9143 - val_loss: 0.2535
Epoch 2/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 51s 26ms/step - accuracy: 0.9283 - loss: 0.2083 - val_accuracy: 0.9101 - val_loss: 0.2504
Epoch 3/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 51s 25ms/step - accuracy: 0.9419 - loss: 0.1674 - val_accuracy: 0.9094 - val_loss: 0.2479
Epoch 4/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 52s 26ms/step - accuracy: 0.9511 - loss: 0.1377 - val_accuracy: 0.9058 - val_loss: 0.2855
Epoch 5/5
1996/1996 ━━━━━━━━━━━━━━━━━━━━ 50s 25ms/step - accuracy: 0.9572 - loss: 0.1157 - val_accuracy: 0.9128 - val_loss: 0.2921


In [26]:
# Save best model
lstm_model.save("../../backend/models/lstm.keras")

In [56]:
# Evaluation

models = {"RNN":rnn_model, "LSTM":lstm_model, "TRANSFORMER":transformer_model}

evaluation = {}

for name, obj in models.items():
    y_pred = (obj.predict(X_test_pad, verbose=0) > 0.5).astype(int).flatten()

    accuracy = accuracy_score(np.array(y_test), y_pred)
    precision = precision_score(np.array(y_test), y_pred)
    recall = recall_score(np.array(y_test), y_pred)
    f1 = f1_score(np.array(y_test), y_pred)
    evaluation[name] = {"Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1}

eval_df = pd.DataFrame(evaluation)
eval_df

,RNN,LSTM,TRANSFORMER
Accuracy,0.904447,0.914189,0.914064
Precision,0.924688,0.899246,0.899805
Recall,0.878780,0.931200,0.930192
F1-Score,0.901150,0.914944,0.914746
